# Galerie données mixtes — 01 · Router les avis clients NovaThread 🟢

> **Étagère optionnelle post-M4** — pas un brief, pas de livrable, pas de note.
> **Autonomie** : 🟢 **résolu** — tu lis, tu exécutes, tu comprends chaque cellule.
> **Durée** : ~2 h
> **Fiches à garder ouvertes** : `fiche_pattern_ML_supervise.md` ·
> `fiche_pattern_texte_NLP.md` · `fiche_preprocessing.pdf` ·
> `fiche_desequilibre_classes.pdf` · `cheatsheet_metriques.md`

## Le contexte

**NovaThread**, e-boutique de mode (fictive), reçoit des centaines d'avis clients
par jour. Léa Fontan, responsable Relation Client :

> « Je veux router automatiquement chaque avis entrant : les clientes
> **insatisfaites** doivent être rappelées par le SAV en priorité, les avis
> **mitigés** partent en file normale, les **satisfaites** alimentent le
> marketing. »

Chaque avis arrive avec des **données tabulaires** (âge de la cliente, rayon,
division) **et du texte libre** (l'avis lui-même). Jusqu'ici tu as traité le
tabulaire (M1, M4) et le texte (M2) **séparément**. Le geste central de ce
notebook : **un seul pipeline scikit-learn qui combine les deux familles de
colonnes** — et une cible à **3 classes déséquilibrées**, évaluée proprement.

> 📦 **Dataset réel** : *Women's E-Commerce Clothing Reviews* (23 486 avis,
> licence CC0, anonymisé — l'enseigne d'origine a été retirée du texte).
> Les avis sont **en anglais** : le geste est identique en français, seuls
> les *stop words* changent.

## Setup

Versions utilisées : `pandas>=2.0`, `scikit-learn>=1.3`, `matplotlib>=3.7`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import sklearn

RANDOM_STATE = 42
print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

## [1] Charger le dataset et fabriquer la cible

Le CSV est téléchargé depuis un miroir public (licence CC0). Si tu es hors
ligne, dépose le fichier dans `data/clothing_reviews.csv` à côté du notebook.

In [ ]:
URL = ("https://raw.githubusercontent.com/AFAgarap/ecommerce-reviews-analysis/"
       "master/Womens%20Clothing%20E-Commerce%20Reviews.csv")

try:
    df = pd.read_csv(URL, index_col=0)
except Exception as err:
    print(f"Téléchargement impossible ({err}) — lecture du CSV local.")
    df = pd.read_csv("data/clothing_reviews.csv", index_col=0)

# noms de colonnes en snake_case, plus faciles à manipuler
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print(df.shape)
df.head(3)

La note (`rating`, 1 à 5 étoiles) sert à fabriquer la **cible métier** de Léa,
en 3 classes :

| Classe | Règle | Action SAV |
|---|---|---|
| `insatisfaite` | note ≤ 2 | rappel prioritaire |
| `mitigée` | note = 3 | file normale |
| `satisfaite` | note ≥ 4 | rien (marketing) |

In [ ]:
def vers_satisfaction(note: int) -> str:
    if note <= 2:
        return "insatisfaite"
    if note == 3:
        return "mitigée"
    return "satisfaite"

df["satisfaction"] = df["rating"].apply(vers_satisfaction)
df["satisfaction"].value_counts(normalize=True).round(3)

### ⚠️ Chasse aux fuites AVANT toute chose

Trois colonnes doivent sortir immédiatement — relis le pourquoi, c'est le
réflexe anti-fuite vu en M4-B1 :

- **`rating`** : c'est la cible déguisée (on vient de la transformer en `satisfaction`) ;
- **`recommended_ind`** : « recommanderiez-vous ce produit ? » — quasi-équivalent
  de la cible, le modèle n'aurait plus rien à apprendre ;
- **`positive_feedback_count`** : nombre de votes « avis utile » reçus **après**
  publication. Au moment où le SAV doit router un avis **entrant**, cette
  information n'existe pas encore → **fuite temporelle**.

In [ ]:
df = df.drop(columns=["rating", "recommended_ind", "positive_feedback_count"])
df.columns.tolist()

## [2] Explorer (EDA mini)

Trois questions minimum avant de modéliser : **manquants ? déséquilibre ?
cardinalités ?** (cf. `fiche_pattern_preparation_donnees.md`).

In [ ]:
print("— Manquants —")
print(df.isna().sum())
print()
print("— Cardinalités des colonnes catégorielles —")
for col in ["clothing_id", "division_name", "department_name", "class_name"]:
    print(f"{col:20s} {df[col].nunique():5d} modalités")

In [ ]:
df["satisfaction"].value_counts().plot.bar(rot=0, title="Cible : 3 classes déséquilibrées")
plt.ylabel("nombre d'avis")
plt.tight_layout()

Ce qu'on retient :

1. **845 avis sans texte** (`review_text` manquant). On ne peut pas les jeter :
   en production, le SAV recevra aussi des avis « note seule, sans commentaire ».
   → on impute par **chaîne vide** : `TfidfVectorizer` produira un vecteur nul
   et le modèle s'appuiera sur le tabulaire pour ces lignes.
2. La cible est **déséquilibrée** (~78 % / 12 % / 10 %) — et la classe rare est
   justement celle qui intéresse Léa. Accuracy interdite comme boussole,
   `f1_macro` comme métrique de référence (cf. `fiche_desequilibre_classes.pdf`).
3. `clothing_id` (1 206 modalités) et `class_name` (20) sont à **haute
   cardinalité** : un OneHot naïf exploserait. On les laisse de côté ici —
   c'est l'objet du notebook 03 de cette galerie.

On en profite pour créer une petite feature honnête : la **longueur de
l'avis** (disponible dès la soumission, aucun risque de fuite).

In [ ]:
df["review_text"] = df["review_text"].fillna("")
df["longueur_avis"] = df["review_text"].str.len()
df[["review_text", "longueur_avis"]].head(3)

## [3] Découper train / test — stratifié, AVANT toute transformation

Le test est **scellé** jusqu'à l'étape [5]. `stratify=y` garantit que les
3 classes gardent leurs proportions dans les deux jeux — indispensable avec
une classe à 10 %.

In [ ]:
from sklearn.model_selection import train_test_split

colonnes_num = ["age", "longueur_avis"]
colonnes_cat = ["division_name", "department_name"]
colonne_txt = "review_text"

X = df[colonnes_num + colonnes_cat + [colonne_txt]]
y = df["satisfaction"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)
y_train.value_counts(normalize=True).round(3)

## [4] Le geste central : UN pipeline pour DEUX familles de colonnes

Un `ColumnTransformer` avec **trois branches** :

| Branche | Colonnes | Traitement |
|---|---|---|
| `num` | âge, longueur | imputation médiane + standardisation |
| `cat` | division, rayon | imputation mode + OneHot (`handle_unknown="ignore"`) |
| `txt` | texte de l'avis | TF-IDF (5 000 termes max) |

> ⚠️ **Piège n°1 du mixte** : la branche texte reçoit le nom de colonne comme
> **chaîne** (`"review_text"`), PAS comme liste (`["review_text"]`).
> `TfidfVectorizer` attend une série 1D de textes ; avec une liste, le
> `ColumnTransformer` lui passe un tableau 2D → erreur incompréhensible.
> C'est LE symptôme classique du premier pipeline mixte.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def fabrique_preparation(avec_tabulaire: bool = True, avec_texte: bool = True) -> ColumnTransformer:
    """Assemble les branches de préparation — utile pour comparer des scénarios de données."""
    branches = []
    if avec_tabulaire:
        branches.append(("num", Pipeline([
            ("imputation", SimpleImputer(strategy="median")),
            ("echelle", StandardScaler()),
        ]), colonnes_num))
        branches.append(("cat", Pipeline([
            ("imputation", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), colonnes_cat))
    if avec_texte:
        branches.append(("txt", TfidfVectorizer(max_features=5000, min_df=3,
                                                stop_words="english"), colonne_txt))
    return ColumnTransformer(branches)


preparation = fabrique_preparation()
preparation

### Trois candidats, mêmes folds, même métrique

Le rituel de M4-B1, transposé : un **plancher** (`Dummy` — *mon modèle
apprend-il quelque chose ?*), un **modèle simple** (régression logistique —
*le complexe est-il justifié ?*), un **ensemble** (RandomForest, ta
connaissance de M1). Validation croisée **stratifiée** 5 folds — avec une
classe à 10 %, une CV non stratifiée pourrait produire des folds quasi vides
sur `insatisfaite`.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

candidats = {
    "plancher (Dummy)": DummyClassifier(strategy="most_frequent"),
    "régression logistique": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "random forest": RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                            n_jobs=-1, random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

lignes = []
for nom, modele in candidats.items():
    pipe = Pipeline([("preparation", fabrique_preparation()), ("modele", modele)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring="f1_macro")
    lignes.append({
        "modèle": nom,
        "f1_macro (CV)": round(scores["test_score"].mean(), 3),
        "écart-type": round(scores["test_score"].std(), 3),
        "temps fit (s)": round(scores["fit_time"].mean(), 1),
    })

pd.DataFrame(lignes).set_index("modèle")

Lis le tableau comme en M4-B1 : perf **et** stabilité **et** coût. La
régression logistique fait ici (au moins) aussi bien que la forêt pour un
temps d'entraînement bien moindre — sur du TF-IDF haute dimension, les
modèles linéaires sont des références solides (cf. `fiche_pattern_texte_NLP.md`).
**Réflexe grille C4** : à performance comparable, prends le plus simple.

## [5] Évaluer sur le test scellé — et LIRE la matrice 3×3

On fige le choix (régression logistique), on ré-entraîne sur tout le train,
on ouvre le test **une seule fois**.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

meilleur = Pipeline([
    ("preparation", fabrique_preparation()),
    ("modele", LogisticRegression(max_iter=2000, class_weight="balanced")),
])
meilleur.fit(X_train, y_train)
y_pred = meilleur.predict(X_test)

print(classification_report(y_test, y_pred, digits=3))

In [ ]:
ordre = ["insatisfaite", "mitigée", "satisfaite"]
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=ordre,
                                        ax=ax, colorbar=False)
ax.set_title("Matrice de confusion — test scellé")
plt.tight_layout()

### Comment lire une matrice 3×3 (le geste qui change tout)

- **Lignes = vérité, colonnes = prédiction.** La diagonale, c'est le juste.
- Toutes les erreurs ne se valent pas : une cliente **insatisfaite prédite
  satisfaite** (case en haut à droite) ne sera **jamais rappelée** — c'est
  l'erreur métier grave. Une satisfaite prédite insatisfaite coûte juste un
  coup de fil inutile.
- Repère la case grave, compte-la, rapporte-la au total de la ligne : c'est
  le **taux de clientes perdues** — une métrique métier qu'aucun `f1_macro`
  ne montre.

Le notebook **02** de cette galerie est entièrement consacré à *réduire* cette
case (poids de classes, seuils de décision).

### 🤔 Question réflexive — d'où vient la performance ?

Ton pipeline mélange tabulaire et texte. Mais **lequel des deux porte
l'information ?** C'est le geste « scénarios de données » : on entraîne le
même modèle sur des sous-ensembles de colonnes et on compare. Tu le
réutiliseras tel quel sur ton cas d'usage (et pour l'analyse avec/sans
variables sensibles, cf. canvas §4.3).

In [ ]:
from sklearn.metrics import f1_score

scenarios = {
    "texte seul": fabrique_preparation(avec_tabulaire=False),
    "tabulaire seul": fabrique_preparation(avec_texte=False),
    "mixte (complet)": fabrique_preparation(),
}

lignes = []
for nom, prep in scenarios.items():
    pipe = Pipeline([("preparation", prep),
                     ("modele", LogisticRegression(max_iter=2000, class_weight="balanced"))])
    pipe.fit(X_train, y_train)
    lignes.append({"scénario": nom,
                   "f1_macro (test)": round(f1_score(y_test, pipe.predict(X_test),
                                                     average="macro"), 3)})

pd.DataFrame(lignes).set_index("scénario")

Verdict typique : **le texte porte l'essentiel du signal**, le tabulaire seul
fait à peine mieux que le plancher, et le mixte apporte un (petit) plus —
notamment pour les 845 avis sans texte, où seul le tabulaire peut parler.
Retiens la **méthode**, pas le chiffre : sur un autre cas, le rapport de force
peut s'inverser.

## [6] Persister le pipeline COMPLET

On sauvegarde le pipeline entier (préparation + modèle), jamais le modèle
seul : à l'inférence, un avis brut doit pouvoir entrer tel quel.

In [ ]:
import joblib

joblib.dump(meilleur, "modele_avis_novathread.joblib", compress=3)
recharge = joblib.load("modele_avis_novathread.joblib")

texte = "Terrible quality, the fabric ripped after one wash. Very disappointed."
nouvel_avis = pd.DataFrame([{
    "age": 34,
    "longueur_avis": len(texte),
    "division_name": "General",
    "department_name": "Dresses",
    "review_text": texte,
}])
print("Prédiction :", recharge.predict(nouvel_avis)[0])
print("Probabilités :", dict(zip(recharge.classes_,
                                 recharge.predict_proba(nouvel_avis)[0].round(3))))

## 📝 Verdict — à remettre à Léa Fontan

À rédiger toi-même en 4-5 lignes (comme en M1-B1) : modèle retenu et pourquoi,
`f1_macro` obtenu, taux de clientes insatisfaites détectées, la limite
principale (les insatisfaites manquées) et le renvoi vers le réglage des
coûts d'erreur (notebook 02).

## 🔎 Ce que tu viens de revoir

- **Pipeline mixte** : `ColumnTransformer` à 3 branches (num / cat / texte) —
  le piège de la colonne texte passée en chaîne, pas en liste.
- **Anti-fuite** : cible déguisée (`rating`), quasi-cible (`recommended_ind`),
  fuite temporelle (`positive_feedback_count`).
- **Texte manquant** : imputation par chaîne vide, le tabulaire prend le relais.
- **CV stratifiée** (`StratifiedKFold`) : obligatoire quand une classe est rare.
- **Matrice 3×3** : lignes = vérité, et toutes les erreurs ne se valent pas.
- **Scénarios de données** : le même modèle sur des sous-ensembles de colonnes,
  pour savoir d'où vient la performance.

## ⭐ Pour aller plus loin (optionnel)

- Ajoute `title` comme **deuxième branche texte** (oui, deux `TfidfVectorizer`
  peuvent coexister dans un `ColumnTransformer`) — gagne-t-on quelque chose ?
- Passe le TF-IDF en bigrammes (`ngram_range=(1, 2)`) : effet sur le f1_macro
  et sur le temps ?
- `class_name` (20 modalités) tient encore en OneHot — mais `clothing_id`
  (1 206 modalités) ? Réponse dans le **notebook 03** (haute cardinalité).